In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer

print("Загрузка модели эмбеддингов...")
embedder = SentenceTransformer("all-MiniLM-L6-v2") # Модель all-MiniLM-L6-v2 работает только с текстом на английском языке, имеет размерность 384

client = QdrantClient(path="../qdrant_storage") # Инициализируем Qdrant на диске. Это значит, что база данных будет храниться в папке qdrant_storage на диске. Если вы хотите использовать облачный Qdrant, используйте QdrantClient(url="https://your-qdrant-instance.com", api_key="your_api_key")

collection_name = "documents"
# vector_size = 384

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

c:\Users\Yernar\Desktop\NLP\etc2\venv3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загрузка модели эмбеддингов...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5902.95it/s]


True

In [5]:
from datasets import load_dataset

# Потоковая загрузка (streaming=True) без сохранения файла на диск
document = load_dataset("rag-datasets/rag-mini-wikipedia", 'text-corpus', split="passages", streaming=True)

In [ ]:
LIMIT = 1000  # Для наглядности возьмем первые 1000 пассажей из потока
points = []
for idx, sample in enumerate(document):
    if idx >= LIMIT:
        break
    
    # Документ/пассаж из датасета
    text = sample.get("passage", sample.get("text", ""))
    passage_id = sample.get("id", idx)

    vector = embedder.encode(text).tolist()

    point = PointStruct(
        id=idx,
        vector=vector,
        payload={
            "text": text,
            "passage_id": passage_id
        }
    )
    points.append(point)

client.upsert(collection_name=collection_name, points=points)

print(f"Успешно заиндексировано и сохранено записей в Qdrant: {len(points)}")

Успешно заиндексировано и сохранено записей в Qdrant: 1000


In [ ]:
query_text = "Location of Uruguay"
query_text1 = "Who is Michael Faraday"
query_text2 = "Who is Millard Fillmore"
print(f"Поисковый запрос: '{query_text2}'")

query_vector = embedder.encode(query_text2).tolist() # Превращаем в вектор

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5 # Выполняем векторный поиск 5 самых похожих записей в Qdrant
)

retrieved_texts = []
print("\n--- Найденный контекст ---")
for hit in search_results.points:
    score = round(hit.score, 4) # Косинусное сходство (чем ближе к 1.0, тем похоже)
    text = hit.payload.get("text", "")
    id = hit.payload.get("passage_id", "None")

    print(f"[Score: {score}, id: {id}]\n{text}")

    retrieved_texts.append(text)

Поисковый запрос: 'Who is Millard Fillmore'

--- Найденный контекст ---
[Score: 0.6913, id: 194]
Millard Fillmore postage stamp
[Score: 0.6442, id: 153]
Engraving of Millard FillmoreIn 1828, Fillmore was elected to the New York State Assembly on the Anti-Masonic ticket, serving for one term, from 1829 to 1831. He was later elected as a Whig (having followed his mentor Thurlow Weed into the party) to the 23rd Congress in 1832, serving from 1833 to 1835. He was re-elected in 1836 to the 25th Congress, to the 26th and to the 27th Congresses and serving from in total from 1833 to 1843, declining to be a candidate for re-nomination in 1842.
[Score: 0.6327, id: 189]
*ESPN anchor Neil Everett often makes references to Millard Fillmore while hosting Sportscenter.
[Score: 0.6085, id: 148]
Millard Fillmore (January 7, 1800   March 8, 1874) was the thirteenth President of the United States, serving from 1850 until 1853, and the last member of the Whig Party to hold that office. He was the second 

In [22]:
retrieved_texts

['Millard Fillmore postage stamp',
 'Engraving of Millard FillmoreIn 1828, Fillmore was elected to the New York State Assembly on the Anti-Masonic ticket, serving for one term, from 1829 to 1831. He was later elected as a Whig (having followed his mentor Thurlow Weed into the party) to the 23rd Congress in 1832, serving from 1833 to 1835. He was re-elected in 1836 to the 25th Congress, to the 26th and to the 27th Congresses and serving from in total from 1833 to 1843, declining to be a candidate for re-nomination in 1842.',
 '*ESPN anchor Neil Everett often makes references to Millard Fillmore while hosting Sportscenter.',
 "Millard Fillmore (January 7, 1800   March 8, 1874) was the thirteenth President of the United States, serving from 1850 until 1853, and the last member of the Whig Party to hold that office. He was the second Vice President to assume the Presidency upon the death of a sitting President, succeeding Zachary Taylor who died of acute gastroenteritis. Fillmore was never

In [23]:
context_block = "\n---\n".join(retrieved_texts) # Метод join() объединяет список retrieved_texts в одну строку, разделяя элементы строкой "\n---\n"

# Формируем финальный промпт, который отправляется в LLM (например, в Qwen2.5 / Llama-3 / GPT-4)
rag_prompt = f"""Используй только следующий контекст, чтобы ответить на вопрос. Если в контексте нет ответа, скажи "Информация не найдена".

Контекст из базы знаний:
{context_block}

Пользовательский вопрос: {query_text1}
Ответ:"""

print("\n--- СФОРМИРОВАННЫЙ RAG-ПРОМПТ ДЛЯ LLM ---")
print(rag_prompt)


--- СФОРМИРОВАННЫЙ RAG-ПРОМПТ ДЛЯ LLM ---
Используй только следующий контекст, чтобы ответить на вопрос. Если в контексте нет ответа, скажи "Информация не найдена".

Контекст из базы знаний:
Millard Fillmore postage stamp
---
Engraving of Millard FillmoreIn 1828, Fillmore was elected to the New York State Assembly on the Anti-Masonic ticket, serving for one term, from 1829 to 1831. He was later elected as a Whig (having followed his mentor Thurlow Weed into the party) to the 23rd Congress in 1832, serving from 1833 to 1835. He was re-elected in 1836 to the 25th Congress, to the 26th and to the 27th Congresses and serving from in total from 1833 to 1843, declining to be a candidate for re-nomination in 1842.
---
*ESPN anchor Neil Everett often makes references to Millard Fillmore while hosting Sportscenter.
---
Millard Fillmore (January 7, 1800   March 8, 1874) was the thirteenth President of the United States, serving from 1850 until 1853, and the last member of the Whig Party to hold

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Настраиваем бесплатную локальную эмбеддинг-модель (BAAI/bge-small-en-v1.5)
print("Инициализация эмбеддера...")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. Имитируем загрузку документов (в реальном проекте можно использовать SimpleDirectoryReader("./data"))
documents = load_dataset("rag-datasets/rag-mini-wikipedia", 'text-corpus', split="passages", streaming=True)

# 3. Индексация данных (LlamaIndex автоматически разбивает текст на чанки,
# вычисляет эмбеддинги и сохраняет их во внутренний векторный индекс)
print("Создание векторного индекса...")
index = VectorStoreIndex.from_documents(documents)

# 4. Превращаем индекс в поисковый движок (Query Engine)
# По умолчанию он найдет релевантные чанки и подготовит контекст для модели
query_engine = index.as_query_engine(similarity_top_k=2)

# 5. Выполняем поиск релевантного контекста под вопрос пользователя
user_query = "За сколько дней нужно предупредить об отпуске?"
print(f"\nВопрос: {user_query}")

# Извлекаем найденные чанки из базы данных
response = query_engine.retrieve(user_query)

print("\n--- Найденный контекст из базы знаний ---")
for node in response:
    print(f"[Score: {round(node.score, 4)}]: {node.text}")

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("Инициализация эмбеддера...")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

documents = load_dataset("rag-datasets/rag-mini-wikipedia", 'text-corpus', split="passages", streaming=True)

print("Создание векторного индекса...")
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine(similarity_top_k=2)

user_query = "За сколько дней нужно предупредить об отпуске?"
print(f"\nВопрос: {user_query}")

response = query_engine.retrieve(user_query)

print("\n--- Найденный контекст из базы знаний ---")
for node in response:
    print(f"[Score: {round(node.score, 4)}]: {node.text}")